In [ ]:
from pathlib import Path

import plotly.graph_objects as go
import polars as pl

from eye_tracking.columns import Column
from eye_tracking.reader import TOBII_COLUMNS, read_raw_data

In [ ]:
raw = read_raw_data(Path("data/1_rawdata.csv"), column_mapping=TOBII_COLUMNS, normalize=True)
raw

In [ ]:
print(f"Samples: {raw.height:,}")
print(f"Duration: {(raw[Column.Timestamp][-1] - raw[Column.Timestamp][0]) / 1_000_000:.2f} s")

In [ ]:
raw.describe()

In [ ]:
left_valid = ((pl.col(Column.LeftX) > 0) & (pl.col(Column.LeftY) > 0) & (pl.col(Column.LeftDiameter) > 0))
right_valid = ((pl.col(Column.RightX) > 0) & (pl.col(Column.RightY) > 0) & (pl.col(Column.RightDiameter) > 0))
validity = raw.select([left_valid.alias("left_valid"), right_valid.alias("right_valid")])

validity.group_by(["left_valid", "right_valid"]).len().sort(["left_valid", "right_valid"], descending=True)


In [ ]:
from eye_tracking.fuse_eyes import fuse_raw_data, EyeFusionProtocol

fused_any = fuse_raw_data(raw, EyeFusionProtocol.Any)
fused_both = fuse_raw_data(raw, EyeFusionProtocol.Both)
fused_aligned = fuse_raw_data(raw, EyeFusionProtocol.Aligned)

In [ ]:
both_valid = raw.filter(left_valid & right_valid)
x_offset = both_valid.select((pl.col(Column.LeftX) - pl.col(Column.RightX)).mean()).item()
y_offset = both_valid.select((pl.col(Column.LeftY) - pl.col(Column.RightY)).mean()).item()

print(f"Mean X offset (left - right): {x_offset:.2f}")
print(f"Mean Y offset (left - right): {y_offset:.2f}")

In [ ]:
missing_one_eye = raw.filter(left_valid ^ right_valid)
missing_one_eye.select([Column.Timestamp, Column.LeftX, Column.RightX, Column.LeftY, Column.RightY]).head(20)

In [ ]:
start = missing_one_eye[Column.Timestamp][0]
end = start + 500_000

raw_window = raw.filter(pl.col(Column.Timestamp).is_between(start, end)).with_columns(
    pl.when(pl.col(Column.LeftX) > 0)
    .then(pl.col(Column.LeftX))
    .otherwise(None)
    .alias(Column.LeftX),
    pl.when(pl.col(Column.RightX) > 0)
    .then(pl.col(Column.RightX))
    .otherwise(None)
    .alias(Column.RightX),
)
aligned_window = fused_aligned.filter(pl.col(Column.Timestamp).is_between(start, end))

traces = [
    (raw_window, Column.LeftX, "Left eye", "#1f77b4", 1),
    (raw_window, Column.RightX, "Right eye", "#ff7f0e", 1),
    (aligned_window, Column.EyeX, "Aligned fusion", "#2ca02c", 3),
]

fig = go.Figure()
for df, column, name, color, width in traces:
    fig.add_trace(
        go.Scatter(
            x=df[Column.Timestamp],
            y=df[column],
            mode="lines+markers",
            name=name,
            line=dict(color=color, width=width),
        )
    )

fig.update_layout(
    title="Eye fusion around a missing-eye segment",
    xaxis_title="Timestamp",
    yaxis_title="X [px]",
    template="plotly_white",
)

fig.show()
